# Submission — arm **C-lr1e3** (🥇 best arm)

`seed 7` · `lr 1e-3` · `eff-batch 64` · `2 epochs`

The only model that beats the constant string. Submit this one first.

## Predicted leaderboard

| | Token F1 | ROUGE-L | **pred LB** |
|---|---|---|---|
| **C-lr1e3** (dev-300, beam 4) | **0.2576** | **0.1776** | **0.5800** |
| Constant string *(currently #1)* | 0.2669 | 0.1564 | 0.57849 |

`LB ≈ 0.4672 + 0.3·TokenF1 + 0.2·ROUGE-L` — reproduces submission 1 exactly (0.57849).

## ⚠️ Before running
| Setting | Value |
|---|---|
| Accelerator | **GPU T4** (x2 is fine — only GPU 0 is used). **P100 is sm_60 and cannot run this.** |
| Internet | **On** |
| Inputs | competition · `farhanishraqq/nascenia-code` · `farhanishraqq/nascenia-ckpts` |

## What this does
1. Loads **only** the `C_lr1e3` checkpoint from the shared checkpoint dataset
2. **Re-scores 300 dev rows** and asserts the result matches `0.2576` — this is an
   integrity check that the intended weights actually loaded, not a tuning step
3. Generates `submission.csv` (`id,output`) on the 1,000 test rows

**Decoding is byte-identical to the training-time eval**: `beam 4 · min_new 80 ·
max_new 320 · length_penalty 1.0`. Do not change it. All three arm notebooks use the
same decoder so the three leaderboard scores are directly comparable — that is what
calibrates the formula above for model output, which so far is fitted to a constant
string only.

**MBR is not used here.** It is the next lever, tested separately on whichever arm wins.


### Submission protocol
Do **not** submit from inside this notebook. Run it, then submit `submission.csv` from the
**Output** tab yourself.

In [ ]:
# ══ 1 — hardware gate ═══════════════════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]} — Kaggle's PyTorch has no sm_60 kernels. Use GPU T4."
print("✅ hardware OK")

In [ ]:
# ══ 2 — pinned libs (trap #00: Kaggle ships transformers 5.0.0, which breaks T5) ══
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers", transformers.__version__, "| normalizer OK:", normalize("হেলো,  নাসেনিয়া ডকে"))

In [ ]:
# ══ 3 — locate code, data, and THIS ARM's checkpoint ════════════════════════
import glob, os, shutil, sys
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/04_decode.py", recursive=True)
assert hits, "Attach Add Input -> Datasets -> farhanishraqq/nascenia-code"
CODE = os.path.dirname(hits[0])

raw = glob.glob("/kaggle/input/**/test.csv", recursive=True)
assert raw, "Attach the Nascenia AI Hackathon competition"
RAW = os.path.dirname(raw[0])

# Pin the checkpoint by NAME. The dataset holds all three arms, and a generic
# "find any config.json" glob would silently pick whichever sorted first — i.e.
# submit the wrong model under this notebook's title.
ARM_DIR = "C_lr1e3"
cands = [os.path.dirname(c) for c in glob.glob("/kaggle/input/**/config.json", recursive=True)
         if ARM_DIR in c and glob.glob(os.path.dirname(c) + "/*.safetensors")]
assert len(cands) == 1, (
    f"expected exactly one {ARM_DIR} checkpoint, found {cands}.\n"
    "Attach Add Input -> Datasets -> farhanishraqq/nascenia-ckpts")
CKPT = cands[0]

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")

# 04_decode.py used to derive its data path from its own location (NOTEBOOKS/ -> ../DATA/
# PROCESSED). Copied to /kaggle/working/code/ that resolves to a directory that does not
# exist, so every hosted decode died on the first read. Fail here, in seconds, rather than
# after the model has loaded.
import subprocess
_h = subprocess.run(["python", "04_decode.py", "--help"], cwd="/kaggle/working/code",
                    capture_output=True, text=True).stdout
assert "--data-dir" in _h, (
    "STALE CODE DATASET: 04_decode.py has no --data-dir. Re-push farhanishraqq/nascenia-code, "
    "then Add Input -> Datasets and pick the newest version.")
print("✅ code dataset is current (04_decode.py supports --data-dir)")

print("CODE:", CODE, "\nRAW :", RAW, "\nCKPT:", CKPT)
print("  ", sorted(os.listdir(CKPT)))

In [ ]:
# ══ 4 — rebuild the identical frozen dev split ══════════════════════════════
# seed 42 / dev-size 5000 governs the DATA SPLIT. It is NOT the training seed and must
# never change — it is what makes this dev number comparable to the training run's.
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

## 5 — Integrity check: does this checkpoint reproduce 0.2576?

300 dev rows, beam 4 — the same rows and decoder the training run evaluated on.
A mismatch means the wrong weights loaded, not that the model changed. ~6 min.

In [ ]:
# ══ 5 — dev re-score ════════════════════════════════════════════════════════
import subprocess, shlex

cmd = (f"python 04_decode.py --ckpt {shlex.quote(CKPT)} --split dev --limit 300 "
       f"--data-dir /kaggle/working/processed "
       f"--mode beam --num-beams 4 --min-new-tokens 80 --max-new-tokens 320 "
       f"--length-penalty 1.0 --no-bertscore --record /kaggle/working/dev.json")
print(cmd, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "dev decode failed"

In [ ]:
# ══ 6 — verify against the recorded training result ═════════════════════════
import json

d = json.load(open("/kaggle/working/dev.json"))["dev"]
f1, rl = d["token_f1"], d["rouge_l"]
EXP_F1, EXP_RL = 0.2576, 0.1776
lb = 0.4672 + 0.3 * f1 + 0.2 * rl

print(f"{'':10s} {'TokenF1':>9s} {'ROUGE-L':>9s} {'pred LB':>9s}")
print(f"{'recorded':10s} {EXP_F1:9.4f} {EXP_RL:9.4f} {0.4672+0.3*EXP_F1+0.2*EXP_RL:9.4f}")
print(f"{'now':10s} {f1:9.4f} {rl:9.4f} {lb:9.4f}")
print(f"{'delta':10s} {f1-EXP_F1:+9.4f} {rl-EXP_RL:+9.4f}")
print(f"\nmean output {d['mean_pred_tokens']:.1f} tokens (references ~100)")

# 0.02 is generous: the training eval and this one build the 300-row subset the same
# way, so they should agree closely. A large gap means the wrong checkpoint.
assert abs(f1 - EXP_F1) < 0.02, (
    f"CHECKPOINT MISMATCH: expected TokenF1 ~{EXP_F1:.4f}, got {f1:.4f}. "
    "This is probably not the C-lr1e3 checkpoint — check cell 3 before submitting.")
print("\n✅ checkpoint verified — this is C-lr1e3")

if f1 < 0.2669:
    print(f"\n⚠️  Token F1 {f1:.4f} is still below the constant string's 0.2669.")
    print(f"   Predicted LB {lb:.4f} vs the constant's 0.57849 — ROUGE-L is what")
    print(f"   decides it, since a constant matches no word order.")

## 7 — Generate the submission on the 1,000 test rows
Same decoder as cell 5. ~15 min.

In [ ]:
# ══ 7 — test decode -> submission.csv ═══════════════════════════════════════
import subprocess, shlex

cmd = (f"python 04_decode.py --ckpt {shlex.quote(CKPT)} --split test "
       f"--data-dir /kaggle/working/processed "
       f"--mode beam --num-beams 4 --min-new-tokens 80 --max-new-tokens 320 "
       f"--length-penalty 1.0 --no-bertscore "
       f"--out /kaggle/working/submission.csv --record /kaggle/working/test_run.json")
print(cmd + "\n" + "=" * 70, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "test decode failed"

In [ ]:
# ══ 8 — sanity checks (a malformed submission wastes a slot) ════════════════
import pandas as pd, glob

sub = pd.read_csv("/kaggle/working/submission.csv")
test = pd.read_csv(glob.glob("/kaggle/input/**/test.csv", recursive=True)[0])

problems = []
if len(sub) != 1000:                       problems.append(f"expected 1000 rows, got {len(sub)}")
if list(sub.columns) != ["id", "output"]:  problems.append(f"columns are {list(sub.columns)}, must be ['id','output']")
if sub["id"].duplicated().any():           problems.append("duplicate ids")
if set(sub["id"]) != set(test["id"]):      problems.append("id set does not match test.csv")
if sub["output"].isna().any():             problems.append("null outputs")
if (sub["output"].astype(str).str.strip() == "").any(): problems.append("empty outputs")

print(f"rows {len(sub)} | cols {list(sub.columns)} | unique ids {sub['id'].nunique()}")
print(f"mean output {sub['output'].astype(str).str.split().str.len().mean():.1f} tokens (references ~100)")
print(("\n❌ " + "; ".join(problems)) if problems else "\n✅ all checks passed — submission.csv ready")
sub.head(3)

---
## After this run

**Submit `submission.csv` from the Output tab yourself.**

Then record in `PREDICTIONS.md`: predicted **0.5800**, the actual score, and the delta.

The delta is the whole point. `LB ≈ 0.4672 + 0.3·F1 + 0.2·RL` is currently fitted to a
**constant string only**. These three arm submissions span Token F1 0.2365 → 0.2576, so
together they show whether the formula holds for model-generated text — and every
downstream decision (which arm to ensemble, whether MBR helped) leans on trusting dev
over the leaderboard.

Archive as `NOTEBOOKS/(score)_nascenia-submit-c-lr1e3/` per CLAUDE.md.
